<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_pinn_plate_switched_on_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 — The Plate Switched On

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.2 · Transient Heat**

The plate of Ex_08.1 is cold, at the coolant's temperature, and its power is
switched on. How fast does it warm, and when is it steady? It is the heat
equation: the steady problem with one more term, the heat the plate stores.
You solve it three ways - a finite-element reference, finite differences, and
a physics-informed network with the start and the channel built in - and
compare them on accuracy and on time. Then the same network finds the plate's
diffusivity from four thermocouples.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | how the reference is built, the three meshes side by side, and finite differences for an agreed 0.20 K | — |
| **5** | a PINN with the start and the channel built in, and the comparison | the trial function, the residual |
| **6** | the inverse problem: the diffusivity from four thermocouples | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About eight minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

The aluminium plate of Ex_08.1 - 100 × 100 mm, a coolant channel at its centre,
insulated outer edges; the channel is the hole of the lecture's slides - sits at the coolant's 40 °C. At $t = 0$ its electronics
are switched on and generate 1 MW/m³. What is the temperature at every point
over the next 80 seconds, and how long before the plate is as hot as it gets?

## 2 · The data

| | |
|---|---|
| plate | 100 × 100 mm, aluminium alloy 6061, $k$ = 167 W/(m·K), $\rho c_p$ = 2.43 MJ/(m³K) |
| diffusivity | $\alpha = k/(\rho c_p)$ = 6.87e-05 m²/s |
| heat generated | $Q$ = 1 MW/m³, evenly, from $t = 0$ |
| cooling channel | an ellipse of 36 × 22 mm at the centre, its wall held at 40 °C |
| outer edges, start | insulated; the whole plate at 40 °C at $t = 0$ |
| window | 0 to 80 s |

## 3 · The physics

The heat that is generated and not conducted away is **stored**, and the
temperature rises (L8.2's first section):

$$\rho c_p\,T_t = k\,(T_{xx} + T_{yy}) + Q$$

with $T$ = 40 °C on the channel and at the start, and no flux through the
edges. The equation is **parabolic**: it has a direction in time, needs **one**
initial field, and smooths as it goes. Heat crosses the plate in a time
$L^2/\alpha$ = 145.51 s, which says what window to look at before anything is
solved.

**Scaled units.** Lengths over $L$, the rise above the coolant over
$\Delta T = QL^2/k$ = 59.88 K, and time over the 80 s window, $\tau = t/80$:

$$\theta_\tau = C\,(\theta_{xx} + \theta_{yy} + 1), \qquad C = \frac{\alpha\cdot 80\ \text{s}}{L^2} = 0.55$$

with $\theta = 0$ on the channel and at $\tau = 0$. Everything below is solved
in these units and printed in kelvin and seconds. The cell below prints the
data and draws the plate, with the four thermocouples section 6 uses.

In [ ]:
pb.describe_problem()
L_mm = pb.L_PLATE * 1e3
t = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.plot([0, L_mm, L_mm, 0, 0], [0, 0, L_mm, L_mm, 0], "--", color="tab:blue", label="insulated edges")
ax.plot(L_mm * (pb.HOLE["xc"] + pb.HOLE["a"] * np.cos(t)), L_mm * (pb.HOLE["yc"] + pb.HOLE["b"] * np.sin(t)),
        color="tab:orange", label="channel wall, 40 °C")
ax.plot(pb.SENSORS[:, 0] * L_mm, pb.SENSORS[:, 1] * L_mm, "k^", ms=7, label="thermocouples")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, a diffusivity of 6.87e-05 m²/s,
145.51 s for heat to cross the plate and a scaled diffusivity of 0.55; and the
plate with its channel and four thermocouples.

---

## 4 · The reference, and finite differences

**Two classical methods, and what differs between them.** Both replace the
field by its values at a finite number of nodes and end in one linear system
for every time step.
They differ in what they ask of the equation.

* **Finite differences** keep the equation as it stands and replace every
  derivative by a difference between neighbouring nodes of a **square grid**
  of spacing $h$:
  $\theta_{xx} \approx (\theta_{i+1,j} - 2\theta_{i,j} + \theta_{i-1,j})/h^2$.
  The equation is asked to hold **at every node**. The grid cannot follow a
  curved wall: a node is either in the channel or in the plate, so the wall
  becomes a **staircase**.
* **Finite elements** cut the plate into **triangles**. The unknowns are the
  values at the corners, and inside a triangle the field is a flat facet. The
  equation is not asked to hold at points but **on average**: it is multiplied
  by a small tent-shaped weight round each node and integrated over the plate,
  once for the tent $v$ of every node. Triangles can be laid along any curve, so **nodes sit on the wall**,
  and an insulated edge needs nothing: it is what the integral gives when
  nothing is prescribed.

The physics-informed network of section 5 is a third way: one smooth formula
for the whole plate, the equation asked at scattered points, and an optimiser
in place of the linear solve.

**In time both do the same thing**: the window is cut into steps, and each
step is taken by the Crank-Nicolson rule, which uses the average of the old
and the new field. So the two differ in space only.

**The reference is not an exact solution.** No formula solves the plate with
an elliptical channel, switched on or not. A finite-element answer, refined in
space and in time until it stops moving, stands in for one, and is called the
**reference**. `pb.reference(spacing, steps)` builds it in seven steps. The
first five are those of Ex_08.1, with one matrix more, for the heat stored:

1. **Nodes.** A square grid of the chosen spacing fills the plate. The grid
   nodes within half a spacing of the channel are dropped, and nodes are laid
   **on the channel wall** at equal steps of arc length.
2. **Triangles.** A Delaunay triangulation joins the nodes, and the triangles
   whose centre lies in the channel are removed.
3. **Tents.** Each node $i$ has a tent $v_i$: 1 at the node, 0 at every other
   node, flat on each triangle. At every instant the answer is
   $\theta \approx \sum_i \theta_i(\tau)\, v_i$: the unknowns are the node values,
   and now they change in time.
4. **Two matrices per triangle.** Conduction, $\int \nabla v_i \cdot \nabla v_j\, dA$,
   as in Ex_08.1. And **storage**, $\int v_i\, v_j\, dA$, which is the
   triangle's area over 12 times
   $\begin{pmatrix} 2 & 1 & 1 \\ 1 & 2 & 1 \\ 1 & 1 & 2 \end{pmatrix}$: the heat a
   node's tent stores, shared with its neighbours.
5. **Assemble.** The element matrices are added into two sparse matrices, $K$
   for conduction and $M$ for storage. The wall nodes are held at
   $\theta = 0$, and the insulated edges need nothing. The heat equation
   becomes $M\,\dot\theta = C\,(f - K\theta)$: one ordinary differential
   equation per node, all coupled to their neighbours.
6. **March in time.** From $\theta = 0$, each step of length $\Delta\tau$ solves
   $(M + \tfrac{1}{2}\Delta\tau\, C K)\,\theta^{\text{new}} = (M - \tfrac{1}{2}\Delta\tau\, C K)\,\theta^{\text{old}} + \Delta\tau\, C f$,
   the Crank-Nicolson rule. The matrix on the left is the same at every step,
   so it is factorised **once**, and each of the 320 steps is then a cheap
   forward and back substitution.
7. **Read anywhere.** 41 instants are stored, 2 s apart. Between the nodes
   the answer is the flat facet of its triangle; between the instants,
   `pb.reference_at` takes the nearest stored one.

The cell below does steps 3 to 6 on a coarse mesh with 40 steps, so you can
see one tent, the two element matrices and the march in time.

In [ ]:
coarse = pb.reference(1 / 25, 40)                            # 4 mm spacing and 40 steps of 2 s, coarse enough to see
P, tri = coarse["P"], coarse["tri"]                          # its nodes, and the three nodes of every triangle
x, y = P[tri, 0], P[tri, 1]                                  # the corners of every triangle
area = 0.5 * np.abs((x[:, 1] - x[:, 0]) * (y[:, 2] - y[:, 0]) - (x[:, 2] - x[:, 0]) * (y[:, 1] - y[:, 0]))
b = np.stack([y[:, 1] - y[:, 2], y[:, 2] - y[:, 0], y[:, 0] - y[:, 1]], 1) / (2 * area[:, None])   # d(tent)/dx
c = np.stack([x[:, 2] - x[:, 1], x[:, 0] - x[:, 2], x[:, 1] - x[:, 0]], 1) / (2 * area[:, None])   # d(tent)/dy
Ke = area[:, None, None] * (b[:, :, None] * b[:, None, :] + c[:, :, None] * c[:, None, :])          # step 4: conduction
Me = area[:, None, None] * (np.ones((3, 3)) + np.eye(3))[None] / 12.0                               # step 4: storage
rows, cols = np.repeat(tri, 3, axis=1).ravel(), np.tile(tri, (1, 3)).ravel()   # where each entry goes in the big matrices
K = sp.coo_matrix((Ke.ravel(), (rows, cols)), shape=(len(P), len(P))).tocsr()  # step 5: conduction, assembled
M = sp.coo_matrix((Me.ravel(), (rows, cols)), shape=(len(P), len(P))).tocsr()  # step 5: storage, the same pattern
print(f"{len(P)} nodes, {len(tri)} triangles; K and M are {K.shape[0]} x {K.shape[1]}, "
      f"{K.nnz / K.shape[0]:.2f} non-zeros per row")
e = np.argmin(np.hypot(x.mean(1) - 0.1, y.mean(1) - 0.1))    # a triangle in the open plate, away from the channel
print("conduction, one triangle (each row sums to zero):"); print(np.round(Ke[e], 2))
print("storage, the same triangle, divided by its area / 12:"); print(np.round(Me[e] * 12 / area[e], 2))

i = np.argmin(np.hypot(P[:, 0] - 0.72, P[:, 1] - 0.50))      # a node near the right end of the channel
v = np.zeros(len(P)); v[i] = 1.0                             # step 3: its tent - 1 there, 0 at every other node
hot = pb.kelvin(coarse["theta"].max(axis=1))                 # step 6: the hottest node's rise after every step
a = np.linspace(0, 2 * np.pi, 200)
wall_x, wall_y = 100 * (pb.HOLE["xc"] + pb.HOLE["a"] * np.cos(a)), 100 * (pb.HOLE["yc"] + pb.HOLE["b"] * np.sin(a))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
tc = ax[0].tripcolor(P[:, 0] * 100, P[:, 1] * 100, tri, v, shading="gouraud", cmap="magma")   # flat on each triangle
plt.colorbar(tc, ax=ax[0], shrink=0.8, label="the tent $v_i$")
ax[0].triplot(P[:, 0] * 100, P[:, 1] * 100, tri, lw=0.4, color="0.6")
ax[0].plot(wall_x, wall_y, color="tab:orange", lw=2)
ax[0].set_xlim(55, 85); ax[0].set_ylim(35, 65); ax[0].set_aspect("equal")
ax[0].set_xlabel("x  [mm]"); ax[0].set_ylabel("y  [mm]")
ax[0].set_title("the tent of one node: 1 there, 0 at its neighbours", fontsize=10)
ax[1].plot(coarse["tau"] * pb.T_END, pb.T_COOLANT + hot, "o-", ms=3, color="tab:red")
ax[1].set_xlabel("t  [s]"); ax[1].set_ylabel("hottest point  [°C]")
ax[1].set_title("the march in time: one dot per step, one sparse solve each", fontsize=10)
plt.tight_layout(); plt.show()

**What you should see.** The 4 mm mesh of Ex_08.1, 652 nodes and 1180
triangles, with about 6.6 non-zeros in a row of $K$ and of $M$: both couple a
node only to its neighbours. The conduction matrix of a right-angled triangle
is the one of Ex_08.1,
$\begin{pmatrix} 0.5 & -0.5 & 0 \\ -0.5 & 1 & -0.5 \\ 0 & -0.5 & 0.5 \end{pmatrix}$,
and the storage matrix is $\begin{pmatrix} 2 & 1 & 1 \\ 1 & 2 & 1 \\ 1 & 1 & 2 \end{pmatrix}$
times the area over 12. On the left, one tent. On the right, the march: the
hottest point rises fast at first and then levels off, one dot for each of the
40 steps. At the reference's resolution, L/200 and 320 steps, the same seven
steps give 38 009 nodes and a step of 0.25 s.

**Three ways to cover the plate.** The cell below draws, at about the same
density, the mesh of the reference, the grid of finite differences and the
points of the network of section 5, whose points also carry a time.

In [ ]:
g = np.linspace(0, 1, 26); MX, MY = np.meshgrid(g, g)        # the square grid at the same 4 mm spacing
held = pb.ellipse_phi(np.c_[MX.ravel(), MY.ravel()]).reshape(MX.shape) <= 0   # nodes the grid puts in the channel
pts_np = pb.sample_spacetime(4000)                           # the network's collocation points (x, y, tau): section 5 trains on these
edge_np = np.vstack([q for q, _ in pb.sample_edges_spacetime(200)])   # and its points on the four insulated edges
fig, ax = plt.subplots(1, 3, figsize=(14.5, 6.0))
ax[0].triplot(P[:, 0] * 100, P[:, 1] * 100, tri, lw=0.4, color="0.45")
ax[0].set_title("finite elements: triangles, nodes on the wall", fontsize=10)
ax[1].plot(MX * 100, MY * 100, lw=0.4, color="0.45"); ax[1].plot(MX.T * 100, MY.T * 100, lw=0.4, color="0.45")
ax[1].plot(MX[held] * 100, MY[held] * 100, "s", ms=3, color="tab:blue", label="nodes held at 40 °C: the staircase")
ax[1].set_title("finite differences: a square grid", fontsize=10)
sc = ax[2].scatter(pts_np[:, 0] * 100, pts_np[:, 1] * 100, c=pts_np[:, 2] * pb.T_END, s=2, cmap="viridis",
                   label="4000 points: the equation")       # coloured by the time each point is asked at
ax[2].plot(edge_np[:, 0] * 100, edge_np[:, 1] * 100, ".", ms=3, color="tab:blue", label="800 points: the insulated edges")
plt.colorbar(sc, ax=ax[2], shrink=0.7, label="t of the point  [s]")
ax[2].set_title("PINN: points in space and time, no mesh", fontsize=10)
for q in ax:
    q.plot(wall_x, wall_y, color="tab:orange", lw=2, label="the channel wall, 40 °C")   # the true wall
    q.set_aspect("equal"); q.set_xlim(-3, 103); q.set_ylim(-3, 103); q.set_xlabel("x  [mm]")
    q.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1)
ax[0].set_ylabel("y  [mm]"); plt.tight_layout(); plt.show()
print(f"network points before 20 s: {100 * np.mean(pts_np[:, 2] < 0.25):.0f} %   (a quarter of the window)")

**What you should see.** The same plate three times. On the left the corners
of the triangles lie on the orange wall. In the middle the wall cuts through
the squares, and the blue nodes are the ones the grid holds at 40 °C: the
channel it solves for has corners. On the right there is no mesh and no time
step: each point is a place **and an instant**, coloured by its time, half of
them in the first quarter of the window, where the field changes fastest; the
edge points are at random instants too, and **no point lies on the channel
wall or at the start**, because the trial function holds both exactly
(section 5).

**How each is built**, side by side:

| | finite elements (the reference) | finite differences | PINN |
|---|---|---|---|
| what is unknown | the value at each node, at each step | the value at each grid node, at each step | the weights of the network |
| between the unknowns | flat on each triangle | nothing: nodes only | one smooth formula in $x$, $y$ and $t$ |
| the equation is asked | on average, against each node's tent | at every node, by differences | at 4000 points in space and time, by autograd |
| in time | Crank-Nicolson, 320 steps, one factorisation | the same | no steps: time is an input |
| the start | $\theta = 0$ before the first step | the same | exact: the factor $\tau$ in the trial function |
| the channel wall | nodes on it, held at 40 °C | a staircase of nodes held at 40 °C | exact: $\phi$ in the trial function |
| the insulated edges | nothing to do | a mirrored node | a loss term, with a weight |
| how it is solved | 320 sparse solves | 320 sparse solves | an optimiser, 650 steps |
| how many unknowns | 38 009 at L/200 | 96 664 at 321 × 321 | about 3 300 |

**How good the reference is.** The cell below halves the node spacing and the
time step together, twice, and prints how far each solution is from the next.

In [ ]:
g = np.linspace(0, 1, 101); GX, GY = np.meshgrid(g, g)       # the points the references are compared on
plate = pb.ellipse_phi(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape) > 0   # True in the plate
fields = {}
for n, steps in ((50, 80), (100, 160), (200, 320)):
    t0 = time.perf_counter(); r = pb.reference(1 / n, steps); sec = time.perf_counter() - t0
    fields[n] = (r, np.array([pb.kelvin(pb.reference_at(r, GX[plate], GY[plate], tau)) for tau in r["tau"]]), sec)
    print(f"  spacing L/{n}, {steps} steps: {len(r['P']):6d} nodes, hottest point at 80 s "
          f"{pb.T_COOLANT + pb.kelvin(r['theta'][-1].max()):.2f} °C, {sec:.2f} s")
for a, b in ((50, 100), (100, 200)):
    print(f"  L/{a} against L/{b}: worst difference over the window {np.nanmax(np.abs(fields[a][1] - fields[b][1])):.2e} K")
ref, t_ref = fields[200][0], fields[200][2]                   # the reference from here on, and what it took
e_ref = np.nanmax(np.abs(fields[100][1] - fields[200][1]))    # how far the next coarser solution is from it
TAUS = ref["tau"]                                             # 41 instants, 2 s apart
hot = pb.kelvin(ref["theta"].max(axis=1))                     # the hottest point's rise at each instant
print(f"  the hottest point reaches {hot[-1]:.2f} K at 80 s; half of that after {np.interp(hot[-1] / 2, hot, TAUS) * pb.T_END:.1f} s")

**What you should see.** The hottest point at 49.42 °C after 80 s - 9.42 K
above the coolant, against 9.56 K in the steady state of Ex_08.1, so the plate
is within 2 % of steady - and half of that rise after about 13 s. The
difference between successive solutions falls with every halving, to 9.44e-03 K
between L/100 and L/200: **more than ten times better than the
accuracy agreed below**, so L/200 with 320 steps is the reference.

**Finite differences.** The square grid of the right picture, with the
channel as a staircase, and Crank-Nicolson in time: one sparse solve per step, the matrix
factorised once. **The agreed accuracy is 0.20 K** over the whole window; the
cell below finds the coarsest grid that meets it.

In [ ]:
def fdm_plate(n, steps, keep=40):
    # n x n nodes, `steps` time steps; returns the nodes and the scaled rise at the 41 stored instants
    h = 1.0 / (n - 1)
    g = np.linspace(0, 1, n); X, Y = np.meshgrid(g, g)
    D = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n)).tolil()
    D[0, 1] = 2.0; D[-1, -2] = 2.0                             # a mirrored node: zero gradient across the edge
    D = D.tocsr() / h ** 2
    A = pb.C_SCALED * (sp.kron(sp.identity(n), D) + sp.kron(D, sp.identity(n))).tocsr()   # C times the Laplacian
    free = np.where(pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).ravel() > 0)[0]           # nodes in the plate
    A = A[free][:, free]; I = sp.identity(len(free)); dt = 1.0 / steps
    lu = spla.splu((I - 0.5 * dt * A).tocsc()); B = (I + 0.5 * dt * A).tocsr()             # Crank-Nicolson
    u = np.zeros(len(free)); out = [np.zeros(n * n)]
    for k in range(1, steps + 1):
        u = lu.solve(B @ u + dt * pb.C_SCALED)                 # one step: storage, conduction, source
        if k % (steps // keep) == 0:
            full = np.zeros(n * n); full[free] = u; out.append(full)
    return X, Y, np.array(out).reshape(-1, n, n)

def error_over_time(X, Y, U):
    # worst error against the reference at each stored instant, K, on the nodes in the plate
    m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
    return np.array([np.nanmax(np.abs(pb.kelvin(U[k][m]) - pb.kelvin(pb.reference_at(ref, X[m], Y[m], tau))))
                     for k, tau in enumerate(TAUS)])

TARGET = 0.20                                                 # K: the agreed accuracy
for n, steps in ((81, 80), (161, 160), (321, 320)):
    t0 = time.perf_counter(); X, Y, U = fdm_plate(n, steps); sec = time.perf_counter() - t0
    err = error_over_time(X, Y, U).max()
    print(f"  {n:3d} x {n} nodes, {steps} steps: worst error {err:.2e} K, one run {sec:.2f} s"
          + ("   <- meets 0.20 K" if err <= TARGET else ""))
N_FDM, STEPS = 321, 320                                       # the coarsest grid that meets it

**What you should see.** The error falling slowly, as on the staircase of
Ex_08.1 - 4.62e-01, 3.15e-01 and 1.25e-01 K - and the 321 × 321 grid with 320
steps the first under 0.20 K, the whole window in five to fifteen seconds.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(x, y, \tau)$ and returns the scaled rise. It is four
layers of 32 tanh neurons, trained on 4000 collocation points spread over the
plate and the window - more of them early, where the field changes fastest.

**Why the scaled time and not the time in seconds.** $\tau = t/80\ \text{s}$
runs from 0 to 1, like $x$ and $y$. A network trains best when its inputs are
all of the same size; the time in seconds would run to 80 beside two inputs
that run to 1. Every result is printed in seconds again. (L8.2 writes $\tau$
for a time constant as well. In this notebook $\tau$ is only the scaled time.)

**The start and the channel wall are built in.** The trial function

$$\hat\theta(x, y, \tau) = S\;\tau\;\phi(x, y)\;\mathcal{N}(x, y, \tau)$$

is exactly zero at $\tau = 0$ and exactly zero on the channel wall, whatever
the network does: one factor for each condition. This is **hard
enforcement**: the two conditions hold by construction, for any weights, and
need no loss term. It is also the first of L8.2's remedies for **causality** -
the start cannot be fitted last if it cannot be wrong.

In it $\mathcal{N}$ is the network, $\phi$ the level set of the channel
(zero on its wall, section 3 of Ex_08.1), and $S$ = 0.01 a fixed number, not
trained, there for size only: $\phi$ grows to 27 in the corners while the
answer stays below 0.2, so without $S$ the network would have to return
numbers of a few thousandths, and with it numbers of order one.

**The insulated edges stay a loss term**, with the weight $w_N$ = 10: **soft
enforcement**, because a factor can hold a value, not a slope.

**TODO 1** is the trial function; **TODO 2** the residual. $C$ = 0.55 is the
scaled diffusivity of section 3, $C = \alpha \cdot 80\ \text{s}/L^2$. The
equation $\theta_\tau = C\,(\theta_{xx} + \theta_{yy} + 1)$ is divided by it,
so that the source term is exactly 1 and every term is of order one:
$\hat\theta_\tau / C - (\hat\theta_{xx} + \hat\theta_{yy} + 1)$.

In [ ]:
SCALE = 0.01                                                  # the size of the answer over the size of phi
class Plate(torch.nn.Module):                                 # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_hidden=32, n_layers=4)       # dense tanh network: x, y, tau in
    def forward(self, p):
        tau = p[:, 2:3]
        return SCALE * tau * pb.ellipse_phi(p) * self.net(p)  # the trial function: zero at the start and on the wall

In [ ]:
def residual(model, p, c):
    th = model(p)                                             # the network's scaled rise
    th_tau = grad(th, p)[:, 2:3]                              # the time derivative by autograd; column 2 is tau
    return th_tau / c - (d2(th, p, 0) + d2(th, p, 1) + 1.0)   # the heat equation, divided by c

In [ ]:
W_N = 10.0                                                    # the weight of the insulated edges
pts = to_tensor(pb.sample_spacetime(4000), requires_grad=True)   # collocation points in space and time, differentiable
edges = [(to_tensor(q, requires_grad=True), axis) for q, axis in pb.sample_edges_spacetime(200)]   # 200 on each edge

def edge_loss(model):
    # the gradient along the normal, squared, averaged over the four edges
    return sum(grad(model(q), q)[:, axis:axis + 1].pow(2).mean() for q, axis in edges) / 4

model = Plate().to(DEVICE)                                    # four layers of 32
describe(model.net, n_collocation=len(pts))                   # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, pts, pb.C_SCALED).pow(2).mean() + W_N * edge_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
t0 = time.perf_counter(); X, Y, U = fdm_plate(N_FDM, STEPS); t_fdm = time.perf_counter() - t0   # finite differences
m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
Pn = to_tensor(np.c_[X[m], Y[m]])                             # the same nodes for the network
def pinn_at(tau, net=None):
    with torch.no_grad():  # no gradient tracking: evaluation only
        col = torch.full((len(Pn), 1), float(tau), device=Pn.device, dtype=Pn.dtype)   # the time column, same device
        return to_numpy((net or model)(torch.cat([Pn, col], 1))).ravel()
P_all = np.array([pinn_at(tau) for tau in TAUS])              # the network at the 41 instants
t_pin = middle(lambda: [pinn_at(tau) for tau in TAUS], reps=3)
e_fdm = error_over_time(X, Y, U)
R_all = np.array([pb.reference_at(ref, X[m], Y[m], tau) for tau in TAUS])
e_pin = np.nanmax(np.abs(pb.kelvin(P_all) - pb.kelvin(R_all)), axis=1)

# "time to the answer" counts everything from the problem to the field over the window: for the network, its training
print(f"{'':32s}{'unknowns':>10s}{'hottest at 80 s':>17s}{'worst error':>13s}{'time to the answer':>20s}")
print(f"  {'reference, fitted L/200':30s}{len(ref['P']):10d}{pb.T_COOLANT + hot[-1]:14.2f} °C{e_ref:11.2e} K{t_ref:18.2f} s   (error: the next coarser solution against it)")
print(f"  {'finite differences, 321 x 321':30s}{int(m.sum()):10d}{pb.T_COOLANT + pb.kelvin(U[-1].max()):14.2f} °C{e_fdm.max():11.2e} K{t_fdm:18.2f} s")
print(f"  {'PINN, 4 x 32':30s}{parameter_count(model):10d}{pb.T_COOLANT + pb.kelvin(P_all[-1].max()):14.2f} °C{e_pin.max():11.2e} K{train_time + t_pin:18.2f} s   (training; the trained network then gives the 41 instants in {t_pin:.2f} s)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + hot, "k", lw=2.5, label="reference")
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + pb.kelvin(U.reshape(len(TAUS), -1).max(axis=1)), "--", label="finite differences")
axes[0].plot(TAUS * pb.T_END, pb.T_COOLANT + pb.kelvin(P_all.max(axis=1)), ":", lw=2.5, label="PINN")
axes[0].set_xlabel("t  [s]"); axes[0].set_ylabel("hottest point  [°C]"); axes[0].legend(frameon=False)
axes[1].semilogy(TAUS[1:] * pb.T_END, e_fdm[1:], label="finite differences, 321 x 321")
axes[1].semilogy(TAUS[1:] * pb.T_END, e_pin[1:], label="PINN")
axes[1].axhline(TARGET, color="k", lw=0.8, ls="--", label="agreed 0.20 K")
axes[1].set_xlabel("t  [s]"); axes[1].set_ylabel("worst error over the plate  [K]"); axes[1].legend(frameon=False)
plt.tight_layout(); plt.savefig(cc.output_path("Ex08.2_report.png"), dpi=150); plt.show()

**What you should see.** The three curves of the hottest point on top of each
other, rising to about 49.4 °C. Finite differences are within 1.25e-01 K of the
reference and the network within about 1.13e-01 K - both inside the agreed
0.20 K.

**By time to the answer** the order is the reference (two to three seconds),
finite differences (five to fifteen seconds), and then the network (about two
minutes, all of it training).

**Is the network more exact, and faster? No, on both counts.** Its error is a
little smaller than that of finite differences on this grid, 0.11 against
0.13 K, and a finer grid would pass it; the two are equally good for the
agreed 0.20 K. And it is faster only *after* it has been trained: reading the
41 instants off the trained network takes a third of a second, but that is
reading an answer already paid for. What it has that the grid lacks is the
wall in the right place, with about 3 300 weights against 100 000 nodes.

---

## 6 · The inverse problem: the diffusivity from four thermocouples

Now suppose the plate's heat capacity is **not known**, so neither is its
diffusivity, but four thermocouples recorded the warm-up every 2 s, with
0.05 K of noise. Can the same network find $\alpha$?

It is L8.2's **Inverse Diffusivity** slide. One number is trained beside the
weights: $C = e^{\beta}$, an exponential so that it cannot go negative. It is
**not an output of the network**, and not an input: it is one more number the
optimiser may change, and it appears in the loss through the residual, where
it multiplies the conduction term. The optimiser sees no difference between
it and a weight. The
loss gains one term, the misfit to the readings, and the residual is the one
you wrote, now with the trainable $C$. $\beta$ starts at $C$ = 0.30, which is
45 % too low. Nothing else changes - which is the point.

In [ ]:
tau_s, readings = pb.sensor_readings(ref)                     # 41 instants x 4 thermocouples, K above the coolant, with noise
S_pts = to_tensor(np.array([[x, y, tau] for tau in tau_s for x, y in pb.SENSORS]))   # where and when they were read
S_val = to_tensor(readings.reshape(-1, 1) / pb.DELTA_T)       # the readings, scaled

class Inverse(torch.nn.Module):                               # the plate's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.plate = Plate()                                  # a fresh network, the same trial function
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(0.30))))   # C = exp(beta), starts 45 % too low
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 1000.0                                                  # the weight of the readings
def inverse_loss():
    c = torch.exp(inv.beta)                                   # the diffusivity being identified, always positive
    data = (inv.plate(S_pts) - S_val).pow(2).mean()           # the misfit at the thermocouples
    return residual(inv.plate, pts, c).pow(2).mean() + W_N * edge_loss(inv.plate) + W_D * data
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
C_found = float(torch.exp(inv.beta))
alpha_found = C_found * pb.L_PLATE ** 2 / pb.T_END            # back to m^2/s
print(f"diffusivity: found {alpha_found:.2e} m^2/s, true {pb.ALPHA:.2e} m^2/s, "
      f"off by {100 * (alpha_found / pb.ALPHA - 1):+.2f} %   ({inv_time:.0f} s)")
print(f"heat capacity that implies: {pb.K_PLATE / alpha_found / 1e6:.2f} MJ/(m³K), true {pb.RHO_CP / 1e6:.2f}")

**What you should see.** The diffusivity found within about 1 % of the
true 6.87e-05 m²/s, in about two and a half minutes on a CPU - from four thermocouples and a
start that was 45 % off.

---

## 7 · What the notebook says

* **Both methods meet the agreed 0.20 K**, and they agree on what an engineer
  wants: the plate is within 2 % of its steady temperature after 80 s, and
  half-way there after about 13 s.
* **For the forward problem, the grid methods win on time**: the network
  takes twenty to fifty times longer. They give the whole window in seconds with no training. The
  network needs minutes, and its only advantage is the one of Ex_08.1: the
  channel is exact, not a staircase.
* **The start was never a problem**, because it is built into the trial
  function. A start left as a loss term is the one instant the network fits
  last; that is the causality of L8.2.
* **The inverse problem is where the network earns its training.** The same
  code, with one more trainable number and one more loss term, returned the
  diffusivity. A grid solver would have to be run again and again inside an
  optimiser to do that.
* **Only the warm-up identifies the diffusivity.** Once the plate is steady the
  readings no longer change, and they say nothing about how fast it got there.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08.2_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "What does switching the heat on add to the steady problem of Ex_08.1, and what did the time L²/α = 145.51 s tell you about the window before anything was solved? (-> L8.2 Q1)": """
""",
    "The trial function is exactly zero at the start. Which of the lecture's remedies for causality is that, and where in time was the network's error largest? (-> L8.2 Q2)": """
""",
    "How was the transient checked when it has no exact solution, and why did finite differences need 321 × 321 nodes for 0.20 K? (-> L8.2 Q3)": """
""",
    "In the inverse problem, what was trained besides the weights, why as an exponential, and how close did four thermocouples bring it to the true diffusivity? (-> L8.2 Q4)": """
""",
    "To conclude: finite differences gave the whole window in seconds. For which task in this notebook did the network do something the grid solver could not do directly? (-> L8.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | hottest point at 80 s | worst error | time to the answer |", "|---|---|---|---|---|",
            f"| reference, fitted mesh L/200, 320 steps | {len(ref['P'])} | {pb.T_COOLANT + hot[-1]:.2f} °C | {e_ref:.2e} K (the next coarser solution against it) | {t_ref:.2f} s |",
            f"| finite differences, 321 x 321, 320 steps | {int(m.sum())} | {pb.T_COOLANT + pb.kelvin(U[-1].max()):.2f} °C | {e_fdm.max():.2e} K | {t_fdm:.2f} s |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {pb.T_COOLANT + pb.kelvin(P_all[-1].max()):.2f} °C | {e_pin.max():.2e} K | {train_time + t_pin:.2f} s, all but {t_pin:.2f} s of it training |",
            "", f"Inverse problem: diffusivity found {alpha_found:.2e} m^2/s, true {pb.ALPHA:.2e} m^2/s, "
                f"off by {100 * (alpha_found / pb.ALPHA - 1):+.2f} %, in {inv_time:.0f} s"]
    out = ["# Ex_08.2 — The Plate Switched On", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.2_report.md into Ex08.2_report.pdf, with any figure
# saved as Ex08.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.2A · The cold plate in three dimensions, switched on

**The problem.** The plate of this set was a section, and it was heated evenly. A real cold
plate is heated from above by a chip that covers a small part of it. When the
chip is switched on, the metal under it warms within a second, while the rest
of the plate takes many seconds to follow. The question is how hot the chip
is at every instant, in three space coordinates and time.

| | |
|---|---|
| block | 100 × 100 mm in plan, 10 mm thick, aluminium, 167 W/m·K, 2.43 MJ/m³K |
| chip | 150 W on 20 × 20 mm of the top face, from (25, 50) to (45, 70) mm: 375 kW/m² |
| cooling | the bottom face, a liquid at 40 °C with $h$ = 5000 W/m²K |
| other faces | insulated: the four sides, and the top beside the chip |
| start, window | the block at 40 °C; the chip on at $t$ = 0; 0 to 20 s |

The conductivity is the set's. The heat transfer coefficient and the chip's
power are typical values, sourced before the data is released.

**Why a plain PINN is not enough.** Four inputs. Two time scales in one field: half of the chip's rise arrives in
half a second, and the last tenth takes fifteen. The flux jumps at the chip's
edge. And the walls are conditions on the slope, so only the start can be
built into the trial function.

**What you build.** A PINN for $T(x, y, z, t)$ in scaled units: the cold start built in by a
factor $t$, as in section 5; the chip's flux, the Robin bottom and the
insulated faces as weighted loss terms. You choose how to spread the points in
time - early times need more - and whether to train the window in one piece.

**The ground truth you get.** `MP8.2A_truth.npz`, from `tools/miniprojects/ex082_truth.py`: the exact
solution as a series in space and time (`coldplate3d.py`): the steady field
less the part that has not yet arrived, 300 × 300 × 200 terms, at 23 instants
and six heights. At $t$ = 0 it returns 1.5e-05 K where it should return zero.

**Worked example.** 

| terms | 0.5 s | 2 s | 10 s | 20 s |
|---|---|---|---|---|
| 150 × 150 × 100 | 53.41 °C | 60.39 °C | 65.65 °C | 66.27 °C |
| 300 × 300 × 200 | 53.39 °C | 60.37 °C | 65.63 °C | 66.25 °C |
| 600 × 600 × 400 | 53.40 °C | 60.38 °C | 65.63 °C | 66.26 °C |

The hottest point, under the chip. Steady it is 66.35 °C. **Half of the rise
is there after 0.48 s and 90 % after 4.51 s**; a single time constant,
$\rho c_p H/h$ = 4.86 s, describes the plate as a whole and misses the first
second entirely.

**Requirements.**

* The hottest point within **0.5 K** at every recorded instant from 0.1 to 20 s.
* The temperature within **1 K** everywhere in the block, at every recorded instant.
* The time at which the chip reaches 90 % of its steady rise within **0.3 s**.

**What you hand in.** The hottest point against the ground truth over the window, the error in
space at 0.5 s and at 20 s, the time to 90 %, and what your choice of points in
time bought you over points spread evenly.

### MP8.2B · Finding a hidden flaw by thermography

**The problem.** Section 6 of this set found one diffusivity for the whole plate. A delamination inside a
composite panel holds heat back only where it is. Flash thermography - the
standard non-destructive test - heats the surface with a flash and watches it
cool with an infrared camera: over a flaw the surface stays warmer, for a
while. The question is where the flaw is, how deep, and how large.

| | |
|---|---|
| section | a 40 mm wide cut through a 4 mm laminate, insulated all round |
| laminate | diffusivity 2e-06 m²/s in-plane and 4e-07 m²/s through the thickness |
| flash | 10 kJ/m² into the top 0.1 mm at $t$ = 0 |
| camera | the surface temperature, every 0.02 s for 10 s, across the 40 mm |
| case 1 | a delamination 10 mm wide at 1 mm deep, 0.2 mm thick, its diffusivity a tenth; noise 0.02 K |
| case 2 | 5 mm wide at 1 mm deep; noise 0.02 K |
| case 3 | 5 mm wide at 2 mm deep; noise 0.05 K |

**Why a plain PINN is not enough.** A diffusivity field is far more to recover than section 6's single number,
and the flaw shows at the surface only in a short window after the flash - and
the deeper it is, the later, the weaker and the more spread out.

**What you build.** An inverse PINN: one network for $\theta(x, z, t)$ and one for the
through-thickness diffusivity $\alpha(x, z)$, kept positive, fitted to the
heat equation, the insulated walls, the flash and the camera frames.

**The ground truth you get.** `MP8.2B_readings.npz` to fit and `MP8.2B_truth.npz` to score against,
from the same script: finite volumes with the conductivity set cell by cell,
cells of 0.1 × 0.025 mm and steps of 5 ms.

**Worked example.** Case 1: the surface over the flaw runs up to **1.14 K** warmer than an
undamaged laminate, 2.86 s after the flash, at its centre, $x$ = 20 mm. The
contrast is 1.143, 1.143 and 1.144 K on the three grids - fifty times the
noise.

**Requirements.**

* Case 1: the flaw's edges within **1 mm** and its depth within **0.2 mm**.
* Every case: say whether there is a flaw, and where; case 3 within **2 mm**.
* The surface temperature within **0.05 K** over the 10 s.

**What you hand in.** The recovered diffusivity against the true one in each case, where your network
puts each flaw, and the smallest and deepest flaw it still finds at the given
noise.